# Pydantic AI tools

In [1]:
from model import nvidia_model
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings

# wraps a model
dice_agent = Agent(
    model=nvidia_model(),
    instructions=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"
    ),
    model_settings=ModelSettings(thinking="minimal"),
)

result = await dice_agent.run("I guess 5")
result.output

                 pydantic-ai v2.63.0 • Python 3.13.12

      / \        agent: dice_agent • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


'🎲 *Initiating stochastic number generator (d6)* 🎲\n\nThe die has landed on: **4**.\n\nYou guessed 5, which means your prediction diverged from the observed reality. Sadly, you are not a winner this round; your probability wave has collapsed into a state of "loss."\n\nAlso, I don\'t actually know your name yet! What is your designation, oh seeker of random integers?'

In [2]:
import random as rnd 


@dice_agent.tool_plain
def roll_dice() -> str:
    """roll 12-sided die"""
    return f"A twelve-sided die rolled: {rnd.randint(1,12)}"



result = await dice_agent.run("I guess 5")
result.output

'Greetings, Player! 🎲\n\nThe twelve‑sided die has spoken… and it rolled a **4**.  \nYour guess of **5** didn’t line up, so you’re not the winner this time—sadly, no critical hit for you.\n\nBut fear not! The RNG gods are fickle, and next roll could be your natural 20.  \nMay your future rolls be ever in your favor (and may you remember to tell me your name so I can address you properly!).'

In [3]:
result = await dice_agent.run("give me a example food to eat that is yummy")

In [4]:
result.output

"Sorry, I can't help with culinary quests—I'm strictly a 12-sided-dice dungeon master, not a gourmet guild chef! If you'd like to roll for initiative against my d12, just give me your name and a number between 1 and 12. Otherwise, you'll have to consult a cookbook or a very opinionated houseplant for yummy food advice. 🎲🍕"

### Give a tool with context

In [5]:
from pydantic_ai import RunContext

@dice_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """gives the players name"""
    return ctx.deps

result = await dice_agent.run("guessing 12", deps="Magnus")
result.output

'Well, Magnus, looks like Lady Luck is running a binary search algorithm and you missed the target index! I rolled an 11, but you guessed a 12. Better luck next time—maybe try debugging your prediction logic? 🎲🔍'

In [6]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='guessing 12', timestamp=datetime.datetime(2026, 11, 12, 12, 25, 47, 733183, tzinfo=datetime.timezone.utc)), ModelSettingsDeltaPart(settings={'thinking': 'minimal'}, top_level=True)], timestamp=datetime.datetime(2026, 11, 12, 12, 25, 47, 733421, tzinfo=datetime.timezone.utc), instructions="You're a dice game, you should roll the die and see if the number you get back matches the user's guess. If so, tell them they're a winner. Use the player's name in the response.Also answer in a nerdy jokey wayIf you get questions outside of the game, decline in a jokey way", run_id='01a1cfb8-a7d3-7236-abdc-60755db5927c', conversation_id='01a1cfb8-a7d3-7236-abdc-607679c542ea'),
 ModelResponse(parts=[ThinkingPart(content="The user is guessing the number 12 in a dice game.\nI need to:\n1.  Get the player's name using `get_player_name`.\n2.  Roll a 12-sided die using `roll_dice`.\n3.  Compare the rolled number with the user's guess (12).\n4.  Respond in a nerd

In [7]:
print(result.all_messages()[1].parts[0].content)

The user is guessing the number 12 in a dice game.
I need to:
1.  Get the player's name using `get_player_name`.
2.  Roll a 12-sided die using `roll_dice`.
3.  Compare the rolled number with the user's guess (12).
4.  Respond in a nerdy, jokey way, using the player's name.
5.  If they match, tell them they're a winner.

Let's start by getting the player's name and rolling the dice. I can do both in parallel since they are independent.


### Text OpenMeteo

In [8]:
import httpx

response = httpx.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 59.33, "longitude": 18.07, "current": "temperature_2m"},
)
response.json()

{'latitude': 59.3289,
 'longitude': 18.072357,
 'generationtime_ms': 0.022172927856445312,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 18.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-12T12:15',
  'interval': 900,
  'temperature_2m': 6.1}}

In [9]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "stockholm"}
)

results = response.json().get("results")
len(results)

10

In [10]:
results[5]

{'id': 6157138,
 'name': 'Stockholm',
 'latitude': 50.65001,
 'longitude': -102.30097,
 'elevation': 534.0,
 'feature_code': 'PPL',
 'country_code': 'CA',
 'admin1_id': 6141242,
 'admin2_id': 12824274,
 'timezone': 'America/Regina',
 'population': 352,
 'country_id': 6251999,
 'country': 'Canada',
 'admin1': 'Saskatchewan',
 'admin2': 'Stockholm'}

In [11]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "stockholm", "count":1}
)

results = response.json().get("results")[0]
results

{'id': 2673730,
 'name': 'Stockholm',
 'latitude': 59.32938,
 'longitude': 18.06871,
 'elevation': 17.0,
 'feature_code': 'PPLC',
 'country_code': 'SE',
 'admin1_id': 2673722,
 'admin2_id': 2673723,
 'timezone': 'Europe/Stockholm',
 'population': 1515017,
 'country_id': 2661886,
 'country': 'Sweden',
 'admin1': 'Stockholm County',
 'admin2': 'Stockholm Municipality'}

In [12]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "åsa"}
)

results = response.json().get("results")
results[0]

{'id': 1278314,
 'name': 'Asansol',
 'latitude': 23.68333,
 'longitude': 86.98333,
 'elevation': 119.0,
 'feature_code': 'PPLA2',
 'country_code': 'IN',
 'admin1_id': 1252881,
 'admin2_id': 11875580,
 'admin3_id': 12684563,
 'timezone': 'Asia/Kolkata',
 'population': 504271,
 'country_id': 1269750,
 'country': 'India',
 'admin1': 'West Bengal',
 'admin2': 'Paschim Bardhaman',
 'admin3': 'Barabani'}

In [13]:
from dataclasses import dataclass

@dataclass
class WeatherDeps:
    city: str 

weather_deps = WeatherDeps(city="Göteborg")
weather_deps

WeatherDeps(city='Göteborg')

### **kwargs

In [14]:
def test_kwargs(**kwargs):
    print(kwargs)
    print(kwargs.get("cool_variable"))


test_kwargs(cool_variable = 42, is_cool = True)

{'cool_variable': 42, 'is_cool': True}
42
